# ML-04 — Search Intelligence Data Contract



This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Unit of analysis and time window

The analysis treats each page-level search observation as the unit of analysis. The purpose is to examine observable search, content, and query-level signals associated with declining search performance and use those signals for directional investigation and page prioritization.

The analysis window is taken directly from the available data rather than assumed in advance. The minimum and maximum observation dates will be measured from the dataset in the verification cell below.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 1: verify the observed date window and basic grain

import pandas as pd
from pathlib import Path

# Find the expected dataset without hard-coding a private/client path.
csv_candidates = list(Path(".").rglob("*.csv"))

print("CSV files found:")
for p in csv_candidates:
    print(p)

# Use the first CSV if only one dataset is present.
if not csv_candidates:
    raise FileNotFoundError("No CSV dataset found. Check the repository/data path.")

df = pd.read_csv(csv_candidates[0])

print("\nShape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

CSV files found:
sample_data/mnist_test.csv
sample_data/california_housing_test.csv
sample_data/mnist_train_small.csv
sample_data/california_housing_train.csv

Shape: (9999, 785)

Columns:
['7', '0', '0.1', '0.2', '0.3', '0.4', '0.5', '0.6', '0.7', '0.8', '0.9', '0.10', '0.11', '0.12', '0.13', '0.14', '0.15', '0.16', '0.17', '0.18', '0.19', '0.20', '0.21', '0.22', '0.23', '0.24', '0.25', '0.26', '0.27', '0.28', '0.29', '0.30', '0.31', '0.32', '0.33', '0.34', '0.35', '0.36', '0.37', '0.38', '0.39', '0.40', '0.41', '0.42', '0.43', '0.44', '0.45', '0.46', '0.47', '0.48', '0.49', '0.50', '0.51', '0.52', '0.53', '0.54', '0.55', '0.56', '0.57', '0.58', '0.59', '0.60', '0.61', '0.62', '0.63', '0.64', '0.65', '0.66', '0.67', '0.68', '0.69', '0.70', '0.71', '0.72', '0.73', '0.74', '0.75', '0.76', '0.77', '0.78', '0.79', '0.80', '0.81', '0.82', '0.83', '0.84', '0.85', '0.86', '0.87', '0.88', '0.89', '0.90', '0.91', '0.92', '0.93', '0.94', '0.95', '0.96', '0.97', '0.98', '0.99', '0.100', '0.101',

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*


The fields are grouped according to their role in the ranking-signal investigation.

**Features:** Observable search, content, and query-level measurements that may be examined as signals associated with search-performance changes.

**Label:** The measured search-performance outcome used to represent the investigation target. The label is defined from an observed outcome rather than from a feature that would leak the outcome.

**Context:** Fields used to describe, group, filter, or interpret observations but not treated as predictive signals.

**Excluded:** Identifiers, administrative fields, or fields that would not be appropriate for the analysis. Fields are excluded when they are not analytically meaningful, contain identifying information, or could introduce target leakage.

The exact column assignments are verified against the available dataset schema before analysis.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 2: Inspect every field so the contract is based on the real schema.

print("FIELD INVENTORY")
print("=" * 70)

for i, col in enumerate(df.columns, start=1):
    print(f"{i:2}. {col:<45} dtype={df[col].dtype}")

print("\n\nMISSINGNESS")
print("=" * 70)

missing = (
    df.isna()
      .sum()
      .to_frame("missing_count")
)

missing["missing_pct"] = (
    missing["missing_count"] / len(df) * 100
).round(2)

print(missing.sort_values("missing_pct", ascending=False).to_string())

FIELD INVENTORY
 1. 7                                             dtype=int64
 2. 0                                             dtype=int64
 3. 0.1                                           dtype=int64
 4. 0.2                                           dtype=int64
 5. 0.3                                           dtype=int64
 6. 0.4                                           dtype=int64
 7. 0.5                                           dtype=int64
 8. 0.6                                           dtype=int64
 9. 0.7                                           dtype=int64
10. 0.8                                           dtype=int64
11. 0.9                                           dtype=int64
12. 0.10                                          dtype=int64
13. 0.11                                          dtype=int64
14. 0.12                                          dtype=int64
15. 0.13                                          dtype=int64
16. 0.14                                          dtyp

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

The checks below verify the claims made in the contract rather than assuming them.

The verification covers:

1. dataset row and column counts;
2. date range;
3. missing values;
4. possible duplicate observations;
5. numeric-field ranges;
6. available categorical/context fields.

These checks describe what is observed in the dataset. They do not establish causality.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 3A: Dataset size and date-window verification

print("DATASET SIZE")
print("=" * 70)
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")

# Detect columns containing date-like information.
date_candidates = [
    c for c in df.columns
    if any(term in c.lower() for term in ["date", "day", "time"])
]

print("\nDATE WINDOW CHECK")
print("=" * 70)

date_results = []

for col in date_candidates:
    parsed = pd.to_datetime(df[col], errors="coerce")

    if parsed.notna().sum() > 0:
        date_results.append({
            "column": col,
            "valid_dates": int(parsed.notna().sum()),
            "min_date": parsed.min(),
            "max_date": parsed.max()
        })

if date_results:
    print(pd.DataFrame(date_results).to_string(index=False))
else:
    print("No parseable date column was detected.")

DATASET SIZE
Rows: 9,999
Columns: 785

DATE WINDOW CHECK
No parseable date column was detected.


In [4]:
# Section 3B: Missing-value verification

missing_check = pd.DataFrame({
    "column": df.columns,
    "missing_count": [df[c].isna().sum() for c in df.columns],
})

missing_check["missing_pct"] = (
    missing_check["missing_count"] / len(df) * 100
).round(2)

print("MISSING VALUE CHECK")
print("=" * 70)
print(
    missing_check
    .sort_values(["missing_pct", "column"], ascending=[False, True])
    .to_string(index=False)
)

MISSING VALUE CHECK
column  missing_count  missing_pct
     0              0          0.0
   0.1              0          0.0
  0.10              0          0.0
 0.100              0          0.0
 0.101              0          0.0
 0.102              0          0.0
 0.103              0          0.0
 0.104              0          0.0
 0.105              0          0.0
 0.106              0          0.0
 0.107              0          0.0
 0.108              0          0.0
 0.109              0          0.0
  0.11              0          0.0
 0.110              0          0.0
 0.111              0          0.0
 0.112              0          0.0
 0.113              0          0.0
 0.114              0          0.0
 0.115              0          0.0
 0.116              0          0.0
 0.117              0          0.0
 0.118              0          0.0
 0.119              0          0.0
  0.12              0          0.0
 0.120              0          0.0
 0.121              0          0.0


In [5]:
# Section 3C: Basic grain / duplicate checks

print("GRAIN CHECK")
print("=" * 70)

# Search for likely page/query/date identifiers.
page_candidates = [
    c for c in df.columns
    if any(term in c.lower() for term in ["page", "url", "landing"])
]

query_candidates = [
    c for c in df.columns
    if "query" in c.lower()
]

date_candidates = [
    c for c in df.columns
    if any(term in c.lower() for term in ["date", "day"])
]

print("Possible page fields:", page_candidates)
print("Possible query fields:", query_candidates)
print("Possible date fields:", date_candidates)

# Test likely combinations when available.
key_candidates = []

if page_candidates and date_candidates:
    key_candidates.append(
        [page_candidates[0], date_candidates[0]]
    )

if page_candidates and query_candidates and date_candidates:
    key_candidates.append(
        [page_candidates[0], query_candidates[0], date_candidates[0]]
    )

for keys in key_candidates:
    duplicate_rows = df.duplicated(subset=keys).sum()

    print(
        f"\nKeys: {keys}"
        f"\nDuplicate rows: {duplicate_rows:,}"
        f"\nUnique combinations: {df[keys].drop_duplicates().shape[0]:,}"
    )

if not key_candidates:
    print(
        "\nNo sufficiently clear page/date or page/query/date key "
        "was automatically identified."
    )

GRAIN CHECK
Possible page fields: []
Possible query fields: []
Possible date fields: []

No sufficiently clear page/date or page/query/date key was automatically identified.


In [6]:
# Section 3D: Numeric-field sanity checks

numeric_cols = df.select_dtypes(include="number").columns.tolist()

print("NUMERIC FIELD SANITY CHECK")
print("=" * 70)

if numeric_cols:
    numeric_summary = df[numeric_cols].describe().T

    numeric_summary["missing"] = df[numeric_cols].isna().sum()

    print(numeric_summary.to_string())
else:
    print("No numeric columns were detected.")

NUMERIC FIELD SANITY CHECK
         count        mean         std  min  25%    50%    75%    max  missing
7       9999.0    4.443144    2.895897  0.0  2.0    4.0    7.0    9.0        0
0       9999.0    0.000000    0.000000  0.0  0.0    0.0    0.0    0.0        0
0.1     9999.0    0.000000    0.000000  0.0  0.0    0.0    0.0    0.0        0
0.2     9999.0    0.000000    0.000000  0.0  0.0    0.0    0.0    0.0        0
0.3     9999.0    0.000000    0.000000  0.0  0.0    0.0    0.0    0.0        0
0.4     9999.0    0.000000    0.000000  0.0  0.0    0.0    0.0    0.0        0
0.5     9999.0    0.000000    0.000000  0.0  0.0    0.0    0.0    0.0        0
0.6     9999.0    0.000000    0.000000  0.0  0.0    0.0    0.0    0.0        0
0.7     9999.0    0.000000    0.000000  0.0  0.0    0.0    0.0    0.0        0
0.8     9999.0    0.000000    0.000000  0.0  0.0    0.0    0.0    0.0        0
0.9     9999.0    0.000000    0.000000  0.0  0.0    0.0    0.0    0.0        0
0.10    9999.0    0.00000

In [7]:
# Section 3E: Cardinality/context check

print("CONTEXT / CARDINALITY CHECK")
print("=" * 70)

for col in df.columns:
    unique_count = df[col].nunique(dropna=True)

    if unique_count <= 20:
        print(
            f"{col}: {unique_count} unique non-null values"
        )

CONTEXT / CARDINALITY CHECK
7: 10 unique non-null values
0: 1 unique non-null values
0.1: 1 unique non-null values
0.2: 1 unique non-null values
0.3: 1 unique non-null values
0.4: 1 unique non-null values
0.5: 1 unique non-null values
0.6: 1 unique non-null values
0.7: 1 unique non-null values
0.8: 1 unique non-null values
0.9: 1 unique non-null values
0.10: 1 unique non-null values
0.11: 1 unique non-null values
0.12: 1 unique non-null values
0.13: 1 unique non-null values
0.14: 1 unique non-null values
0.15: 1 unique non-null values
0.16: 1 unique non-null values
0.17: 1 unique non-null values
0.18: 1 unique non-null values
0.19: 1 unique non-null values
0.20: 1 unique non-null values
0.21: 1 unique non-null values
0.22: 1 unique non-null values
0.23: 1 unique non-null values
0.24: 1 unique non-null values
0.25: 1 unique non-null values
0.26: 1 unique non-null values
0.27: 1 unique non-null values
0.28: 1 unique non-null values
0.29: 1 unique non-null values
0.30: 1 unique non-null v

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This dataset supports measurement and directional investigation, but it has important limits.

- The data can show observed associations between available signals and search-performance outcomes; it cannot by itself establish causation.
- Uneven historical coverage can make comparisons less reliable for observations with shorter histories.
- If early observations contain primarily Search Console information while later observations contain additional context, the available evidence is not identical across the full window.
- Overlapping time windows can cause observations to share underlying information, so they should not automatically be treated as independent evidence.
- Missing values can reduce the usable evidence for particular fields or groups.
- Search performance can be affected by factors that are not represented in the dataset, so the observed signals should not be interpreted as a complete explanation of ranking changes.
- The output is intended for page prioritization and decision-support, not as proof that changing a particular feature will cause a ranking improvement.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4: Quantify the main data limitations where possible.

print("DATA LIMITS CHECK")
print("=" * 70)

print(f"Total observations: {len(df):,}")

# Overall missingness
total_cells = df.shape[0] * df.shape[1]
missing_cells = int(df.isna().sum().sum())

print(
    f"Missing cells: {missing_cells:,} "
    f"({missing_cells / total_cells * 100:.2f}% of all cells)"
)

# Columns with missing data
columns_with_missing = int((df.isna().sum() > 0).sum())

print(
    f"Columns containing missing values: "
    f"{columns_with_missing:,} / {len(df.columns):,}"
)

# Constant columns
constant_columns = [
    c for c in df.columns
    if df[c].nunique(dropna=False) <= 1
]

print(
    f"Constant / single-value columns: "
    f"{len(constant_columns):,}"
)

if constant_columns:
    print("Constant columns:", constant_columns)

# Potentially high-cardinality identifier-like columns
identifier_like = []

for col in df.columns:
    unique_ratio = df[col].nunique(dropna=True) / max(len(df), 1)

    if unique_ratio > 0.95:
        identifier_like.append(col)

print(
    f"\nPotential high-cardinality identifier-like fields: "
    f"{len(identifier_like):,}"
)

if identifier_like:
    print(identifier_like)

DATA LIMITS CHECK
Total observations: 9,999
Missing cells: 0 (0.00% of all cells)
Columns containing missing values: 0 / 785
Constant / single-value columns: 116
Constant columns: ['0', '0.1', '0.2', '0.3', '0.4', '0.5', '0.6', '0.7', '0.8', '0.9', '0.10', '0.11', '0.12', '0.13', '0.14', '0.15', '0.16', '0.17', '0.18', '0.19', '0.20', '0.21', '0.22', '0.23', '0.24', '0.25', '0.26', '0.27', '0.28', '0.29', '0.30', '0.31', '0.32', '0.49', '0.50', '0.51', '0.52', '0.53', '0.54', '0.55', '0.56', '0.57', '0.58', '0.59', '0.60', '0.81', '0.82', '0.83', '0.84', '0.85', '0.86', '0.87', '0.111', '0.112', '0.113', '0.114', '0.139', '0.140', '0.167', '0.168', '0.169', '0.196', '0.218', '0.283', '0.306', '0.307', '0.330', '0.331', '0.332', '0.354', '0.355', '0.356', '0.379', '0.403', '0.404', '0.428', '0.452', '0.476', '0.498', '0.499', '0.500', '0.522', '0.523', '0.524', '0.545', '0.546', '0.569', '0.570', '0.571', '0.572', '0.591', '0.592', '0.593', '0.594', '0.595', '0.614', '0.615', '0.616', '

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.